[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch13-model-serving/04-mot-yeu-cau-va-hoa-don.ipynb)

# Một yêu cầu, từ byte trên đường truyền tới hoá đơn

Ba sổ tay trước nhìn dòng yêu cầu ở trạng thái ổn định: hàng đợi, gom batch, bộ nhớ đệm KV. Sổ tay này
đi theo **một** yêu cầu. Nó tới dưới dạng byte, được giải tuần tự hóa, tiền xử lý, chạy qua mô hình rồi
hậu xử lý. Trước đó, mô hình phải được nạp và làm nóng, và tiền xử lý phải khớp với lúc huấn luyện. Cuối
cùng, mọi lựa chọn ấy đổi ra số máy và tiền. Chương 13 cho rất nhiều con số minh hoạ ở mỗi chặng. Ta tính
lại chúng, và đo những gì đo được trên CPU của bạn.

**Bạn sẽ làm:**
1. đo JSON với dạng nhị phân kiểu Protobuf cho 1,000 số thực, rồi tính lại napkin math 1.1;
2. tính lại ngân sách độ trễ của ResNet-50 (bảng 3), định luật Amdahl, DLRM (bảng 4) và thuế phục vụ
   (bảng 5);
3. đo từng pha của một yêu cầu ResNet-50 trên máy này;
4. tìm hai hằng số mà mô hình đơn giản của bảng 6 dùng nhưng không in ra;
5. tính chu kỳ hoạt động của bộ tăng tốc khi xử lý tuần tự và khi pipeline;
6. huấn luyện một mô hình MNIST, phục vụ nó với tiền xử lý lệch, và viết một kiểm thử đối chiếu;
7. đo khởi động nguội trên máy này, tính bảng 8, và mô phỏng một đợt tăng vọt khi bản sao mới khởi động
   nguội;
8. tính bảng 19, bảng 20, phương trình 16 và 17, đo việc gộp chuẩn hóa theo batch vào tích chập, và xem
   dữ liệu hiệu chuẩn không đại diện làm gì;
9. tính chi phí mỗi triệu yêu cầu, đội máy cho 5,000 QPS, và điểm giao nhau giữa CPU và GPU.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import base64
import copy
import gc
import heapq
import io
import json
import warnings

import torch.nn.functional as F
import torchvision
from PIL import Image
from torchvision import datasets

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # thứ tự màu cố định
warnings.filterwarnings("ignore", message="The given NumPy array is not writable")


def thoi_gian(f, lap=50, khoi_dong=5):
    """Trung vị thời gian của f() qua nhiều lần chạy, tính bằng giây."""
    for _ in range(khoi_dong):
        f()
    ts = []
    for _ in range(lap):
        t1 = time.perf_counter()
        f()
        ts.append(time.perf_counter() - t1)
    return float(np.median(ts))


def varint(n):
    """Số nguyên không âm theo varint của Protobuf: 7 bit mỗi byte, bit cao báo còn byte sau."""
    ra = bytearray()
    while True:
        b, n = n & 0x7F, n >> 7
        ra.append(b | 0x80 if n else b)
        if not n:
            return bytes(ra)


THE_PB = (1 << 3) | 2   # trường số 1, kiểu dây 2 (có độ dài): cách Protobuf gửi `repeated float` dạng packed


def ma_hoa_pb(v):
    """Mã hoá một mảng float32 đúng như Protobuf mã hoá trường `repeated float x = 1` dạng packed."""
    du_lieu = np.asarray(v, dtype="<f4").tobytes()
    return bytes([THE_PB]) + varint(len(du_lieu)) + du_lieu


def giai_ma_pb(buf):
    """Đọc byte thẻ, đọc độ dài varint, rồi xem thẳng phần byte còn lại là float32, không chép."""
    assert buf[0] == THE_PB
    i, n, dich = 1, 0, 0
    while True:
        b = buf[i]
        i += 1
        n |= (b & 0x7F) << dich
        dich += 7
        if not b & 0x80:
            break
    return np.frombuffer(buf, dtype="<f4", count=n // 4, offset=i)


def json_gon(v):
    """JSON với mỗi số in ngắn nhất mà vẫn đọc lại đúng float32."""
    return "[" + ",".join(np.format_float_positional(x, unique=True, trim="-") for x in v) + "]"


print(f"varint(4,000) = {varint(4000).hex(' ')} · thẻ trường = {THE_PB:#04x}")

## 1. Tuần tự hóa: JSON hay byte nhị phân

Dữ liệu gửi cho một hệ thống học máy thường là mảng số thực dày đặc, như một vector embedding hay một
tensor ảnh. JSON phải viết mỗi số thành một chuỗi ký tự có độ dài thay đổi, và phía nhận phải đọc từng
byte rồi đổi ngược lại. Protocol Buffers gửi cả mảng dưới dạng byte gọn của chuẩn dấu phẩy động, 4 byte
mỗi số float32. Napkin math 1.1 của chương xét một yêu cầu chứa 1,000 số thực.

Sổ tay không dựa vào thư viện `protobuf`, mà tự mã hoá theo đúng định dạng trên dây của nó để bạn thấy từng
byte, cho một trường `repeated float` dạng packed: một byte thẻ, độ dài viết theo varint, rồi các
byte của mảng. Đọc lại bằng `np.frombuffer` thì không chép gì, gần với ý "đọc thẳng bộ đệm mạng" mà
chương gán cho FlatBuffers.

**Dự đoán:** JSON của 1,000 số thực nặng gấp mấy lần dạng nhị phân? Phân tích nó chậm hơn gấp mấy lần?

In [ ]:
SO_PHAN_TU = sach(1000, nguon="1,000 · 9 KB")
v = np.random.default_rng(42).standard_normal(SO_PHAN_TU).astype(np.float32)  # giả định: embedding phân phối chuẩn

json_f64 = json.dumps(v.tolist())        # Python in mỗi số theo float64, tới 17 chữ số
json_f32 = json_gon(v)
pb = ma_hoa_pb(v)
b64_json = json.dumps({"x": base64.b64encode(v.tobytes()).decode()})
assert np.array_equal(giai_ma_pb(pb), v)
assert np.array_equal(np.array(json.loads(json_f32), dtype=np.float32), v)
print("JSON mặc định:", json_f64[:58] + "…")
print("JSON gọn:     ", json_f32[:58] + "…")
print("Protobuf:     ", pb[:6].hex(" "), "…")
print()
for ten, n in (("JSON, số in theo float64", len(json_f64)), ("JSON, số in gọn theo float32", len(json_f32)),
               ("Base64 của mảng, bọc trong JSON", len(b64_json)), ("nhị phân kiểu Protobuf", len(pb))):
    print(f"{ten:<33} {n:>7,} byte · {n / SO_PHAN_TU:5.2f} byte mỗi số · gấp {n / len(pb):4.2f} lần Protobuf")
print(f"\n9 KB của sách ứng với khoảng {sach(9, nguon='9 KB · 50 μs') * 1000 / SO_PHAN_TU:.0f} byte mỗi số, kể cả dấu phẩy;"
      f" 4 KB ứng với {sach(4, nguon='4 KB · 5 μs') * 1000 / SO_PHAN_TU:.0f} byte mỗi số")
theo_sach("Base64 làm kích thước tăng thêm (%)", 100 * (len(base64.b64encode(v.tobytes())) / v.nbytes - 1),
          sach=33, trich="which adds 33 percent size overhead")

Kích thước của JSON tuỳ vào cách phía gửi in số: cùng một mảng, in theo float64 thì nặng gần gấp đôi in
gọn. Đó là điều chương nói khi viết rằng độ dài của chuỗi phụ thuộc giá trị và định dạng. Dạng nhị phân
thì luôn là 4 byte mỗi số cộng vài byte đầu (thẻ và độ dài). Giờ đo thời gian đọc lại mỗi dạng.

In [ ]:
cach_doc = {
    "JSON in theo float64": lambda: json.loads(json_f64),
    "JSON in gọn": lambda: json.loads(json_f32),
    "JSON in gọn, rồi đổi sang mảng NumPy": lambda: np.array(json.loads(json_f32), dtype=np.float32),
    "Base64 trong JSON, rồi đổi sang mảng": lambda: np.frombuffer(base64.b64decode(json.loads(b64_json)["x"]), np.float32),
    "nhị phân kiểu Protobuf, xem thẳng bộ đệm": lambda: giai_ma_pb(pb),
}
t_doc = {}
for ten, f in cach_doc.items():
    t_doc[ten] = thoi_gian(f, lap=300, khoi_dong=30)
    do_tren_may(f"đọc {SO_PHAN_TU:,} số, {ten}", 1e6 * t_doc[ten], "μs")
chenh = t_doc["JSON in gọn, rồi đổi sang mảng NumPy"] - t_doc["nhị phân kiểu Protobuf, xem thẳng bộ đệm"]
QPS_VD = sach(10_000, nguon="10,000 yêu cầu")
do_tren_may(f"phần một lõi CPU được giải phóng ở {QPS_VD:,} yêu cầu mỗi giây", 100 * QPS_VD * chenh, "%")
print()
json_us, pb_us = sach(50, nguon="9 KB · 50 μs"), sach(5, nguon="4 KB · 5 μs")
theo_sach("thời gian phân tích tiết kiệm mỗi yêu cầu (μs)", json_us - pb_us, sach=45, nguon="5 μs · 45 μs")
theo_sach(f"CPU lấy lại mỗi giây ở {QPS_VD:,} yêu cầu/s (s)", QPS_VD * (json_us - pb_us) / 1e6, sach=0.45, nguon="0.45 giây")
theo_sach("phần của một lõi (%)", 100 * QPS_VD * (json_us - pb_us) / 1e6, sach=45, nguon="0.45 giây · 45 %")

Bảng 5 ghi chi phí tuần tự hóa "tuyến tính theo dữ liệu gửi". Ô sau đo hai cách đọc ở nhiều kích thước,
từ 10 tới 100,000 số.

In [ ]:
co_mang = [10, 100, 1_000, 10_000, 100_000]
do_json, do_pb = [], []
rng = np.random.default_rng(7)
for n in co_mang:
    m = rng.standard_normal(n).astype(np.float32)
    s_json, s_pb = json_gon(m), ma_hoa_pb(m)
    lap = 300 if n <= 1_000 else (30 if n <= 10_000 else 5)
    do_json.append(thoi_gian(lambda: np.array(json.loads(s_json), dtype=np.float32), lap=lap, khoi_dong=2))
    do_pb.append(thoi_gian(lambda: giai_ma_pb(s_pb), lap=lap, khoi_dong=2))
    print(f"{n:>7,} số: JSON {len(s_json):>9,} byte, {1e6 * do_json[-1]:>9.1f} μs · Protobuf {len(s_pb):>7,} byte,"
          f" {1e6 * do_pb[-1]:>6.2f} μs (đo trên máy này)")
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.plot(co_mang, 1e6 * np.array(do_json), "o-", color=MAU[1], lw=2, ms=6, label="JSON in gọn → mảng NumPy")
ax.plot(co_mang, 1e6 * np.array(do_pb), "o-", color=MAU[0], lw=2, ms=6, label="nhị phân kiểu Protobuf")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("số phần tử float32 trong một yêu cầu (thang log)")
ax.set_ylabel("thời gian đọc (μs, thang log)")
ax.set_title("Đọc một mảng số thực từ yêu cầu (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.25, which="both")
plt.tight_layout()
plt.show()

JSON tăng gần như tỉ lệ với số phần tử, vì mỗi byte đều phải đọc và mỗi số đều phải đổi. Dạng nhị phân
chỉ đọc vài byte đầu rồi xem phần còn lại là một mảng, nên ở các kích thước này nó gần như không đổi. Nếu
phía nhận cần một bản chép riêng (để ghi vào, hay để chuyển sang bộ tăng tốc), chi phí ấy là một lần chép
bộ nhớ, vẫn rẻ hơn nhiều so với đổi văn bản ra số. Chương cũng nói lựa chọn còn tuỳ ràng buộc: REST với
JSON hợp với API công khai, cần dễ gỡ lỗi và tương thích rộng, còn gRPC với Protobuf hợp với lưu lượng
tensor nội bộ nhiều QPS.

## 2. Ngân sách độ trễ của một yêu cầu ResNet-50

Chương định nghĩa ngân sách độ trễ là thời gian chia cho các chặng của một yêu cầu dưới một hạn chót đầu
cuối. Đó là hệ tổng bằng không: mỗi mi-li-giây tuần tự hóa hay mạng tiêu mất là một mi-li-giây bớt đi cho
suy luận. Bảng 3 là một phân tích minh hoạ cho một yêu cầu ResNet-50 trên GPU, với tỉ lệ đã làm tròn.

**Dự đoán:** nếu mô hình chạy nhanh gấp 10 lần, cả yêu cầu nhanh lên bao nhiêu lần?

In [ ]:
NGAN_SACH = {  # bảng 3, ms
    "giải mã JPEG": sach(3, trich="JPEG decode | 3 ms | 30%"),
    "đổi kích thước": sach(1, trich="1 ms | 10%"),
    "chuẩn hoá": sach(0.5, trich="Normalize (mean/std) | 0.5 ms | 5%"),
    "chép CPU → GPU": sach(0.5, trich="CPU→GPU copy | 0.5 ms | 5%"),
    "suy luận ResNet-50": sach(5, trich="ResNet-50 forward pass | 5 ms | 50%"),
    "softmax + top-5": sach(0.1, trich="Softmax + top-5 | 0.1 ms | 1%"),
}
TIEN_XU_LY = ("giải mã JPEG", "đổi kích thước", "chuẩn hoá")
tong3 = sum(NGAN_SACH.values())
phan = {k: 100 * ms / tong3 for k, ms in NGAN_SACH.items()}
theo_sach("giải mã JPEG: phần trăm của tổng", phan["giải mã JPEG"], sach=30, trich="JPEG decode | 3 ms | 30%")
theo_sach("đổi kích thước: phần trăm", phan["đổi kích thước"], sach=10, trich="1 ms | 10%")
theo_sach("chuẩn hoá: phần trăm", phan["chuẩn hoá"], sach=5, trich="Normalize (mean/std) | 0.5 ms | 5%")
theo_sach("chép CPU → GPU: phần trăm", phan["chép CPU → GPU"], sach=5, trich="CPU→GPU copy | 0.5 ms | 5%")
theo_sach("suy luận: phần trăm", phan["suy luận ResNet-50"], sach=50, trich="ResNet-50 forward pass | 5 ms | 50%")
theo_sach("softmax + top-5: phần trăm", phan["softmax + top-5"], sach=1, trich="Softmax + top-5 | 0.1 ms | 1%")
tien3 = sum(NGAN_SACH[k] for k in TIEN_XU_LY)
theo_sach("tổng cả yêu cầu (ms)", tong3, sach=10.1, nguon="0.1 ms · 10.1 ms")
theo_sach("phần tiền xử lý (%)", 100 * tien3 / tong3, sach=44.6, nguon="10.1 ms · 44.6 %")
SL_TRT = sach(2, nguon="44.6 % · 2 ms")          # mức suy luận giả định sau TensorRT
tong_trt = tong3 - NGAN_SACH["suy luận ResNet-50"] + SL_TRT
theo_sach("phần tiền xử lý khi suy luận còn 2 ms (%)", 100 * tien3 / tong_trt, sach=63.4, nguon="2 ms · 63.4 %")
print()
theo_sach("tiền xử lý (ms)", tien3, sach=4.5, nguon="4.5 ms · 0.5 ms")
ngoai_mo_hinh = tien3 + NGAN_SACH["chép CPU → GPU"]
theo_sach("tiền xử lý và truyền dữ liệu, phần của tổng (%)", 100 * ngoai_mo_hinh / tong3, sach=49.5, nguon="0.5 ms · 49.5 %")
K_MH = sach(10, nguon="49.5 % · 10 lần")
tong_10 = tong3 - NGAN_SACH["suy luận ResNet-50"] + NGAN_SACH["suy luận ResNet-50"] / K_MH
theo_sach("cả yêu cầu khi mô hình nhanh gấp 10 lần (ms)", tong_10, sach=5.6, nguon="10.1 ms · 5.6 ms")
theo_sach("tăng tốc đầu cuối", tong3 / tong_10, sach=1.8, nguon="5.6 ms · 1.8 lần")
f_mh = NGAN_SACH["suy luận ResNet-50"] / tong3
print(f"trần của định luật Amdahl khi mô hình chiếm {100 * f_mh:.1f} %: {1 / (1 - f_mh):.2f} lần, dù mô hình nhanh tới đâu")

Mô hình đại diện 1.1 áp cùng cách phân tích cho DLRM, một mô hình gợi ý, với ngân sách p99 10 ms. Ở đây
điểm nghẽn dời từ tính toán sang băng thông bộ nhớ: phần lớn thời gian là tra embedding. Bảng 5 thì liệt
kê **thuế phục vụ**, những khoản độ trễ ngoài suy luận mà yêu cầu nào cũng đóng. Chương tính ba khoản ở
thang mi-crô-giây (tuần tự hóa, điều phối, chép dữ liệu) so với một dịch vụ suy luận 5 ms.

In [ ]:
DLRM = {"phân tích yêu cầu (CPU)": sach(0.5, nguon="10 ms · 0.5 ms"),
        "tra embedding (băng thông bộ nhớ)": sach(6, nguon="100 · 6 ms"),
        "lượt truyền xuôi MLP (tính toán)": sach(1.5, nguon="6 ms · 1.5 ms"),
        "xếp hạng và lọc (CPU)": sach(1, nguon="1.5 ms · 1 ms")}
tong4 = sum(DLRM.values())
theo_sach("DLRM: tổng (ms)", tong4, sach=9, nguon="1 ms · 9 ms")
theo_sach("DLRM: phần của MLP (%)", 100 * DLRM["lượt truyền xuôi MLP (tính toán)"] / tong4, sach=17, nguon="9 ms · 17 %")
f_dlrm = DLRM["lượt truyền xuôi MLP (tính toán)"] / tong4
print(f"dưới ngân sách p99 {sach(10, nguon='1.1 · 10 ms')} ms · trần Amdahl nếu chỉ tối ưu MLP: {1 / (1 - f_dlrm):.2f} lần")
print()
THUE = {  # bảng 5, khoảng thấp và cao, μs
    "tuần tự hóa": (sach(50, trich=r"Serialization | 50–500 \(\mu\text{s}\) | Linear with payload"),
                    sach(500, trich=r"Serialization | 50–500 \(\mu\text{s}\) | Linear with payload")),
    "điều phối": (sach(10, trich=r"Dispatch | 10–50 \(\mu\text{s}\) | Constant per batch"),
                  sach(50, trich=r"Dispatch | 10–50 \(\mu\text{s}\) | Constant per batch")),
    "chép dữ liệu": (sach(100, trich=r"Data Copy | 100–500 \(\mu\text{s}\) | Linear with tensor"),
                     sach(500, trich=r"Data Copy | 100–500 \(\mu\text{s}\) | Linear with tensor")),
}
SL5 = sach(5, nguon="5 ms · 50 · 500 μs")
thap = sum(lo for lo, _ in THUE.values()) / 1000 / SL5
cao = sum(hi for _, hi in THUE.values()) / 1000 / SL5
for k, (lo, hi) in THUE.items():
    print(f"{k:<13} {lo:>4}–{hi:<4} μs = {100 * lo / 1000 / SL5:4.1f}–{100 * hi / 1000 / SL5:4.1f} % của 5 ms")
theo_sach("ba khoản mi-crô-giây, đầu thấp (%)", 100 * thap, sach=3.2, nguon="100 · 3.2 · 21 %")
theo_sach("ba khoản mi-crô-giây, đầu cao (%)", 100 * cao, sach=21, nguon="3.2 · 21 %")
mang = (sach(1, trich="Network I/O | 1–5 ms | Linear with payload"), sach(5, trich="Network I/O | 1–5 ms | Linear with payload"))
hang = (sach(0.1, trich="Queuing | 0.1–10 ms | Diverges near capacity"), sach(10, trich="Queuing | 0.1–10 ms | Diverges near capacity"))
print(f"cộng thêm mạng {mang[0]}–{mang[1]} ms và hàng đợi {hang[0]}–{hang[1]} ms: cả khoản thuế là"
      f" {thap * SL5 + mang[0] + hang[0]:.2f}–{cao * SL5 + mang[1] + hang[1]:.2f} ms quanh một lần suy luận {SL5} ms")
print()
SLO_CP, DA_DANH = sach(50, nguon="50 ms · 40 lần"), sach(18, nguon="1.1 · 18 ms")
print(f"checkpoint 1.1: SLO p99 {SLO_CP} ms, đã dành {DA_DANH} ms ngoài mô hình → mô hình còn {SLO_CP - DA_DANH} ms")
theo_sach("checkpoint 1.5: phần còn cho ứng dụng, mô hình và kernel (ms)",
          sach(20, nguon="1.13 · 20 ms") - sach(6, nguon="20 ms · 6 ms"), sach=14, nguon="6 ms · 14 ms")
GAP_P99 = sach(40, nguon="50 ms · 40 lần")
print(f"trung bình {SLO_CP} ms mà ngưỡng p99 gấp {GAP_P99} lần: ngưỡng ấy là {GAP_P99 * SLO_CP / 1000:.0f} giây, và khoảng"
      f" {sach(1, nguon='40 lần · 1 %')} % yêu cầu mất từ chừng ấy trở lên")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw={"width_ratios": [1.35, 1]})
nhom = {"tiền xử lý": (TIEN_XU_LY, MAU[0]), "chép CPU → GPU": (("chép CPU → GPU",), MAU[3]),
        "suy luận": (("suy luận ResNet-50",), MAU[1]), "hậu xử lý": (("softmax + top-5",), MAU[2])}
ca = {"bảng 3": 1.0, "suy luận còn 2 ms\n(TensorRT, giả định)": SL_TRT / NGAN_SACH["suy luận ResNet-50"],
      "mô hình nhanh\ngấp 10 lần": 1 / K_MH}
for y, (ten_ca, he_so) in enumerate(ca.items()):
    trai = 0.0
    for ten_nhom, (pha, mau) in nhom.items():
        for p in pha:
            w = NGAN_SACH[p] * (he_so if p == "suy luận ResNet-50" else 1.0)
            ax1.barh(y, w, left=trai, color=mau, edgecolor="white", linewidth=1.5, height=0.6,
                     label=ten_nhom if (y == 0 and p == pha[0]) else None)
            trai += w
    ax1.text(trai + 0.15, y, f"{trai:.1f} ms · tiền xử lý {100 * tien3 / trai:.1f} %", va="center", fontsize=8)
ax1.set_yticks(range(len(ca)), list(ca))
ax1.invert_yaxis()
ax1.set_xlim(0, 15.5)
ax1.set_xlabel("độ trễ của một yêu cầu (ms)")
ax1.set_title("Ngân sách độ trễ ResNet-50 (bảng 3)")
ax1.legend(loc="lower right", fontsize=8)
k = np.logspace(0, 3, 200)
for f_, ten, mau in ((f_mh, f"ResNet-50, mô hình chiếm {100 * f_mh:.1f} %", MAU[1]),
                     (f_dlrm, f"DLRM, MLP chiếm {100 * f_dlrm:.1f} %", MAU[4])):
    ax2.plot(k, 1 / ((1 - f_) + f_ / k), color=mau, lw=2, label=ten)
    ax2.axhline(1 / (1 - f_), color=mau, lw=1, ls=":")
ax2.plot([K_MH], [tong3 / tong_10], "o", color=MAU[1], ms=8, mec="white", mew=1.5)
ax2.annotate(f"10 lần → {tong3 / tong_10:.1f} lần", (K_MH, tong3 / tong_10), xytext=(16, -22),
             textcoords="offset points", fontsize=8)
ax2.set_xscale("log")
ax2.set_ylim(1, 2.2)
ax2.set_xlabel("mô hình nhanh lên bao nhiêu lần (thang log)")
ax2.set_ylabel("cả yêu cầu nhanh lên bao nhiêu lần")
ax2.set_title("Định luật Amdahl trong phục vụ")
ax2.legend(loc="upper left", fontsize=8)
ax2.grid(alpha=0.25)
plt.tight_layout()
plt.show()

Bên trái: khi mô hình nhanh lên, phần tiền xử lý không đổi nên chiếm ngày càng nhiều. Bên phải: tăng tốc
đầu cuối bị chặn bởi phần nằm ngoài mô hình, khoảng 2 lần cho ResNet-50 và khoảng 1.2 lần cho DLRM. Lời
khuyên của chương là phân tích hiệu năng trước khi tối ưu. Với ResNet-50, điểm nghẽn tiếp theo là giải
mã ảnh trên CPU, còn với DLRM là băng thông bộ nhớ của bảng embedding.

## 3. Đo một yêu cầu ResNet-50 trên máy này

Bảng 3 là số giả định trên GPU. Ở đây ta đo từng pha của cùng đường đi trên CPU của bạn, với một luồng.
Ảnh vào là một JPEG 640×480 tổng hợp (giả định của sổ tay). ResNet-50 lấy từ torchvision với trọng số
ngẫu nhiên (`weights=None`), vì sổ tay không tải gì ngoài bộ dữ liệu. Thời gian của một lượt truyền xuôi
không phụ thuộc giá trị của trọng số. Phần hậu xử lý gồm softmax, top-5 và dựng phản hồi JSON có phiên
bản mô hình, như mục hậu xử lý của chương tả.

**Dự đoán:** trên CPU không có bộ tăng tốc, pha nào chiếm phần lớn thời gian?

In [ ]:
CAO, RONG = 480, 640
yy, xx = np.mgrid[0:CAO, 0:RONG]
anh = np.stack([xx / RONG * 255, yy / CAO * 255, (xx + yy) % 64 * 4], axis=-1)
anh = np.clip(anh + np.random.default_rng(42).normal(0, 20, anh.shape), 0, 255).astype(np.uint8)
bo_dem = io.BytesIO()
Image.fromarray(anh).save(bo_dem, format="JPEG", quality=90)
JPEG = bo_dem.getvalue()
CANH = sach(224, nguon="224 · 1 ms")
TB_IMAGENET = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)   # hằng số chuẩn hoá quen thuộc của ImageNet
DLC_IMAGENET = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
dat_hat_giong()
resnet = torchvision.models.resnet50(weights=None).eval()


def giai_ma():
    return Image.open(io.BytesIO(JPEG)).convert("RGB")


def doi_kich_thuoc(a):
    return a.resize((CANH, CANH), Image.BILINEAR)


def chuan_hoa(a):
    x = torch.from_numpy(np.array(a)).permute(2, 0, 1).float().div(255)
    return ((x - TB_IMAGENET) / DLC_IMAGENET).unsqueeze(0)


def hau_xu_ly(logit):
    p = torch.softmax(logit, dim=1)
    diem, lop = torch.topk(p, 5)
    return json.dumps({"phien_ban_mo_hinh": "resnet50-thu", "top5": [
        {"lop": int(c), "diem": round(float(d), 4)} for d, c in zip(diem[0], lop[0])]})


a = giai_ma()
b = doi_kich_thuoc(a)
x = chuan_hoa(b)
with torch.inference_mode():
    logit = resnet(x)
    pha_do = {
        "giải mã JPEG": thoi_gian(giai_ma, lap=30),
        "đổi kích thước về 224×224": thoi_gian(lambda: doi_kich_thuoc(a), lap=30),
        "chuẩn hoá thành tensor": thoi_gian(lambda: chuan_hoa(b), lap=30),
        "suy luận ResNet-50, batch 1": thoi_gian(lambda: resnet(x), lap=5, khoi_dong=1),
        "hậu xử lý: softmax, top-5, JSON": thoi_gian(lambda: hau_xu_ly(logit), lap=100),
    }
print(f"JPEG {len(JPEG):,} byte → tensor {tuple(x.shape)} → phản hồi {hau_xu_ly(logit)[:70]}…")
for ten, t in pha_do.items():
    do_tren_may(ten, 1000 * t, "ms")
t_mh = pha_do["suy luận ResNet-50, batch 1"]
t_ngoai = sum(pha_do.values()) - t_mh
do_tren_may("phần ngoài mô hình trong cả yêu cầu", 100 * t_ngoai / (t_ngoai + t_mh), "%")
k_nua = t_mh / t_ngoai
do_tren_may("mô hình phải nhanh lên bao nhiêu lần để phần ngoài mô hình chiếm một nửa", k_nua, "lần")

In [ ]:
k = np.logspace(0, 3.3, 300)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(k, 100 * t_ngoai / (t_ngoai + t_mh / k), color=MAU[0], lw=2)
ax.axhline(50, color="0.5", lw=1, ls="--")
ax.plot([k_nua], [50], "o", color=MAU[0], ms=8, mec="white", mew=1.5)
ax.annotate(f"mô hình nhanh {k_nua:.0f} lần", (k_nua, 50), xytext=(10, -18), textcoords="offset points", fontsize=8)
ax.set_xscale("log")
ax.set_ylim(0, 100)
ax.set_xlabel("nếu chỉ phần mô hình nhanh lên k lần (thang log)")
ax.set_ylabel("phần thời gian ngoài mô hình (%)")
ax.set_title("Một yêu cầu ResNet-50: phần ngoài mô hình lớn dần (đo trên máy này)")
ax.grid(alpha=0.25, which="both")
plt.tight_layout()
plt.show()

Trên CPU, lượt truyền xuôi lấn át mọi pha khác, nên tỉ lệ ở đây khác hẳn bảng 3. Đó đúng là điều chương
nhấn mạnh: tỉ lệ giữa các pha là của một tổ hợp phần cứng, không phải của mô hình. Đường cong cho thấy
chuyện gì xảy ra khi phần mô hình được tăng tốc, bằng một bộ tăng tốc hay một môi trường chạy tốt hơn:
phần tiền xử lý trên CPU không đổi và dần thành điểm nghẽn. Chương gọi đó là xu hướng làm trần Amdahl
càng chặt, vì bộ tăng tốc nhanh lên nhanh hơn nhiều so với hiệu năng đơn luồng của CPU.

Chức năng thứ hai của máy chủ suy luận là đổi dữ liệu sang bố trí tensor mà môi trường chạy cần, và
chương nói bố trí nào nhanh hơn (NCHW hay NHWC) tuỳ vào kernel, kiểu dữ liệu và phần cứng. Ô sau đo cả hai
trên máy này.

In [ ]:
resnet_nhwc = copy.deepcopy(resnet).to(memory_format=torch.channels_last)
x_nhwc = x.contiguous(memory_format=torch.channels_last)
with torch.inference_mode():
    t_nchw = thoi_gian(lambda: resnet(x), lap=5, khoi_dong=1)
    t_nhwc = thoi_gian(lambda: resnet_nhwc(x_nhwc), lap=5, khoi_dong=1)
    lech_bt = (resnet(x) - resnet_nhwc(x_nhwc)).abs().max().item()
do_tren_may("ResNet-50, bố trí NCHW", 1000 * t_nchw, "ms")
do_tren_may("ResNet-50, bố trí NHWC (channels_last)", 1000 * t_nhwc, "ms")
print(f"trên máy này, {'NHWC' if t_nhwc < t_nchw else 'NCHW'} nhanh hơn {max(t_nchw, t_nhwc) / min(t_nchw, t_nhwc):.2f} lần;"
      f" đầu ra hai cách lệch nhau tối đa {lech_bt:.1e}")
del resnet_nhwc

## 4. Độ phân giải: phương trình 1 và hai hằng số của bảng 6

Với mô hình chủ yếu là tích chập và nghẽn ở tính toán, phương trình 1 nói thông lượng tỉ lệ nghịch với
bình phương độ phân giải. Kịch bản minh hoạ của chương ghi 3.6 lần thay cho 4 lần khi đi từ 224 lên 448,
"vì có tính cả phụ trội cố định". Chương không nói phụ trội ấy lớn bao nhiêu, nhưng hai con số đủ để suy
ra nó.

In [ ]:
r1, r2 = sach(224, nguon="224 · 448"), sach(448, nguon="224 · 448 · 4 lần")
theo_sach("224 → 448: tỉ số tính toán theo phương trình 1", (r2 / r1) ** 2, sach=4, nguon="448 · 4 lần")
TY_SO = sach(3.6, nguon="4 lần · 3.6 lần")
# T(r) = a + b·(r / 224)²  ⇒  (a + 4b) / (a + b) = 3.6
a_tren_b = ((r2 / r1) ** 2 - TY_SO) / (TY_SO - 1)
print(f"suy ra (sách không nêu): phần cố định chiếm {100 * a_tren_b / (1 + a_tren_b):.1f} % thời gian ở 224×224"
      f" và {100 * a_tren_b / (4 + a_tren_b):.1f} % ở 448×448")

Bảng 6 là một mô hình đơn giản của mật độ tính toán: số phép tính tăng theo bình phương độ phân giải, chia
cho số byte gồm trọng số (cố định) cộng một số hạng kích hoạt (cũng tăng theo bình phương). Bảng in số hạng
kích hoạt và kết quả, nhưng không in số phép tính hay số byte trọng số. Hai dòng của bảng là đủ để giải ra
hai ẩn ấy. Rồi hai dòng còn lại kiểm tra lời giải.

In [ ]:
canh6 = np.array([sach(224, trich=r"\(224{\times}224\) | 12.5 MB | 64.4 FLOP/byte"),
                  sach(384, trich=r"\(384{\times}384\) | 36.7 MB | 137 FLOP/byte"),
                  sach(512, trich=r"\(512{\times}512\) | 65.3 MB | 183.9 FLOP/byte"),
                  sach(640, trich=r"\(640{\times}640\) | 102.0 MB | 218.4 FLOP/byte")], dtype=float)
kich6 = 1e6 * np.array([sach(12.5, trich=r"\(224{\times}224\) | 12.5 MB | 64.4 FLOP/byte"),
                        sach(36.7, trich=r"\(384{\times}384\) | 36.7 MB | 137 FLOP/byte"),
                        sach(65.3, trich=r"\(512{\times}512\) | 65.3 MB | 183.9 FLOP/byte"),
                        sach(102.0, trich=r"\(640{\times}640\) | 102.0 MB | 218.4 FLOP/byte")])
mat6 = np.array([sach(64.4, trich=r"\(224{\times}224\) | 12.5 MB | 64.4 FLOP/byte"),
                 sach(137, trich=r"\(384{\times}384\) | 36.7 MB | 137 FLOP/byte"),
                 sach(183.9, trich=r"\(512{\times}512\) | 65.3 MB | 183.9 FLOP/byte"),
                 sach(218.4, trich=r"\(640{\times}640\) | 102.0 MB | 218.4 FLOP/byte")])
s6 = (canh6 / canh6[0]) ** 2
# mật độ = F·s / (W + A)  ⇒  F·s − mật độ·W = mật độ·A, giải với dòng đầu và dòng cuối
F_khop, W_khop = np.linalg.solve([[s6[0], -mat6[0]], [s6[3], -mat6[3]]], [mat6[0] * kich6[0], mat6[3] * kich6[3]])
print(f"lời giải: {F_khop / 1e9:.3f} tỷ phép tính mỗi ảnh ở 224×224, trọng số {W_khop / 1e6:.2f} MB")

nhan_cong = [0]


def dem(mod, vao, ra):
    if isinstance(mod, torch.nn.Conv2d):
        nhan_cong[0] += ra.numel() * (mod.in_channels // mod.groups) * mod.kernel_size[0] * mod.kernel_size[1]
    elif isinstance(mod, torch.nn.Linear):
        nhan_cong[0] += ra.numel() * mod.in_features


moc = [m.register_forward_hook(dem) for m in resnet.modules() if isinstance(m, (torch.nn.Conv2d, torch.nn.Linear))]
with torch.inference_mode():
    resnet(torch.zeros(1, 3, CANH, CANH))
for m in moc:
    m.remove()
SO_THAM_SO = sum(p.numel() for p in resnet.parameters())
print(f"đếm trên ResNet-50 của torchvision: {nhan_cong[0] / 1e9:.3f} tỷ phép nhân-cộng, {SO_THAM_SO:,} tham số"
      f" = {2 * SO_THAM_SO / 1e6:.2f} MB ở FP16")
print()
F6 = 4.1e9   # giả định: số tròn hay được trích cho ResNet-50, 4.1 tỷ phép mỗi ảnh 224×224
W6 = 2 * 1e6 * sach(25.6, trich="Not feasible (25.6 MB)")   # bản INT8 25.6 MB tức 25.6 triệu tham số; ở FP16 gấp đôi


def kich_hoat(r):
    return kich6[0] * (r / canh6[0]) ** 2


def mat_do(r):
    return F6 * (r / canh6[0]) ** 2 / (W6 + kich_hoat(r))


print(f"với {F6 / 1e9} tỷ phép và {W6 / 1e6:.1f} MB:")
theo_sach("224×224: mật độ tính toán (FLOP/byte)", mat_do(canh6[0]), sach=64.4,
          trich=r"\(224{\times}224\) | 12.5 MB | 64.4 FLOP/byte")
theo_sach("384×384: số hạng kích hoạt (MB)", kich_hoat(canh6[1]) / 1e6, sach=36.7,
          trich=r"\(384{\times}384\) | 36.7 MB | 137 FLOP/byte")
theo_sach("384×384: mật độ tính toán (FLOP/byte)", mat_do(canh6[1]), sach=137,
          trich=r"\(384{\times}384\) | 36.7 MB | 137 FLOP/byte")
theo_sach("512×512: số hạng kích hoạt (MB)", kich_hoat(canh6[2]) / 1e6, sach=65.3,
          trich=r"\(512{\times}512\) | 65.3 MB | 183.9 FLOP/byte")
theo_sach("512×512: mật độ tính toán (FLOP/byte)", mat_do(canh6[2]), sach=183.9,
          trich=r"\(512{\times}512\) | 65.3 MB | 183.9 FLOP/byte")
theo_sach("640×640: số hạng kích hoạt (MB)", kich_hoat(canh6[3]) / 1e6, sach=102.0,
          trich=r"\(640{\times}640\) | 102.0 MB | 218.4 FLOP/byte")
theo_sach("640×640: mật độ tính toán (FLOP/byte)", mat_do(canh6[3]), sach=218.4,
          trich=r"\(640{\times}640\) | 102.0 MB | 218.4 FLOP/byte")

Lời giải rơi đúng vào ResNet-50: khoảng 4.1 tỷ phép mỗi ảnh và khoảng 51 MB trọng số, tức 25.6 triệu
tham số ở FP16. Với hai số tròn ấy, cả bốn dòng của bảng khớp tới chữ số cuối. Đáng để ý hai điều. Thứ
nhất, 4.1 tỷ là số **phép nhân-cộng** mà ta vừa đếm; nếu tính 2 FLOP cho mỗi phép nhân-cộng, như cách đếm
FLOP thường gặp, mọi mật độ trong bảng nhân đôi. Thứ hai, mô hình bỏ qua tensor trung gian, bộ nhớ đệm và
vùng làm việc, nên chương ghi cột phân loại điểm nghẽn là "chưa xác lập". Vượt điểm gãy của một GPU trên
bảng này chưa chứng minh được kernel thật nghẽn ở đâu.

## 5. Chu kỳ hoạt động: tuần tự hay pipeline

Tiền xử lý chạy trên CPU, suy luận chạy trên bộ tăng tốc. Xử lý tuần tự thì mỗi tài nguyên ngồi chờ tài
nguyên kia. Phương trình 2 định nghĩa chu kỳ hoạt động của phần cứng là tổng thời gian tính hữu ích chia
cho tích của thời gian thực với số tài nguyên. Ví dụ của chương: một yêu cầu mất 10 ms, gồm 5 ms trên GPU
và 5 ms trên CPU.

Với thời gian cố định ở mỗi chặng, thông lượng của một pipeline hai chặng là số nhỏ nhất trong ba giới
hạn: số yêu cầu đang xử lý chia cho thời gian của cả vòng, năng lực của GPU, và năng lực của các luồng CPU.

In [ ]:
T_YC = sach(10, nguon="50 · 10 ms")
G = sach(5, nguon="10 ms · 5 ms")    # thời gian trên GPU
C = T_YC - G                         # thời gian trên CPU


def chu_ky_gpu(c, dang_xu_ly, luong_cpu=1, g=G):
    """Phần thời gian GPU bận: thông lượng nhân thời gian GPU của mỗi yêu cầu."""
    thong_luong = np.minimum(np.minimum(dang_xu_ly / (g + c), 1 / g), luong_cpu / c)
    return thong_luong * g


theo_sach("tuần tự: chu kỳ hoạt động của GPU (%)", 100 * chu_ky_gpu(C, 1), sach=50, nguon="5 ms · 50 % · 100 %")
theo_sach("pipeline hai yêu cầu (%)", 100 * chu_ky_gpu(C, 2), sach=100, nguon="50 % · 100 %")
theo_sach("trần tăng tốc khi tiền xử lý chiếm 50 %", 1 / (C / T_YC), sach=2, nguon="100 % · 2 lần")
c = np.linspace(0.5, 20, 300)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for ten, kw, mau in (("tuần tự (một yêu cầu một lúc)", dict(dang_xu_ly=1), MAU[1]),
                     ("pipeline, một luồng CPU", dict(dang_xu_ly=8, luong_cpu=1), MAU[0]),
                     ("pipeline, hai luồng CPU", dict(dang_xu_ly=8, luong_cpu=2), MAU[2])):
    ax.plot(c, 100 * chu_ky_gpu(c, **kw), color=mau, lw=2, label=ten)
ax.axvline(C, color="0.6", lw=1, ls=":")
ax.text(C + 0.3, 6, f"ví dụ của chương: CPU {C} ms, GPU {G} ms", fontsize=8, color="0.3")
ax.set_ylim(0, 105)
ax.set_xlim(0, 20)
ax.set_xlabel("thời gian tiền xử lý trên CPU cho mỗi yêu cầu (ms)")
ax.set_ylabel("chu kỳ hoạt động của GPU (%)")
ax.set_title("Chu kỳ hoạt động của GPU, suy luận 5 ms mỗi yêu cầu")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

Bên trái vạch chấm, CPU nhanh hơn GPU, và pipeline giữ GPU bận hoàn toàn. Bên phải, CPU quá chậm để nuôi
GPU: hệ thống thành nghẽn ở CPU, và tối ưu mô hình thêm không cho thêm chút thông lượng nào. Lúc ấy chỉ
thêm luồng CPU hay chuyển tiền xử lý sang bộ tăng tốc mới giúp. Một hệ nghẽn ở tính toán trên bộ tăng tốc
cũ có thể thành nghẽn ở CPU sau khi nâng cấp, chỉ vì G nhỏ đi còn C giữ nguyên.

## 6. Lệch giữa huấn luyện và phục vụ, và kiểm thử đối chiếu

Chương định nghĩa lệch giữa huấn luyện và phục vụ (training-serving skew) là độ lệch phân phối giữa môi
trường huấn luyện và môi trường suy luận, do logic hay trạng thái không nhất quán. Ví dụ 1.5 của chương có
phía phục vụ tiền xử lý bằng OpenCV, còn phía huấn luyện dùng PIL, với cách nội suy và hằng số chuẩn hoá
khác nhau. Hệ thống vẫn chạy trơn tru, chỉ dự đoán sai đi.

Phần này phỏng theo phần A của phòng thí nghiệm đi kèm chương. Ta dựng một phiên bản nhỏ: ảnh "gốc" 64×64
(giả định của sổ tay: ảnh MNIST phóng lên bằng nội suy bậc ba), một hàm biến đổi của phía huấn luyện (thu
nhỏ về 28×28 bằng PIL, chia 255, chuẩn hoá theo trung bình và độ lệch chuẩn của MNIST), và một mạng
784-256-10 học 3 epoch trên 20,000 ảnh. Rồi ta phục vụ nó bằng vài hàm biến đổi khác, mỗi hàm lệch một
chút.

**Dự đoán:** hàm nào làm mất nhiều độ chính xác nhất? Một kiểm thử đối chiếu so tensor từng điểm có bắt
được những thay đổi không làm mất độ chính xác nào không?

In [ ]:
dat_hat_giong()
CANH_THO, N_HL, N_KT = 64, 20_000, 5_000     # giả định của sổ tay
tap_hl = datasets.MNIST(DATA, train=True, download=True)
tap_kt = datasets.MNIST(DATA, train=False, download=True)


def lam_anh_tho(x):
    return np.stack([np.asarray(Image.fromarray(a).resize((CANH_THO, CANH_THO), Image.BICUBIC)) for a in x.numpy()])


tho_hl, tho_kt = lam_anh_tho(tap_hl.data[:N_HL]), lam_anh_tho(tap_kt.data[:N_KT])
y_hl, y_kt = tap_hl.targets[:N_HL], tap_kt.targets[:N_KT]


def thu_nho_pil(tho, bo_loc=Image.BILINEAR):
    return np.stack([np.asarray(Image.fromarray(a).resize((28, 28), bo_loc)) for a in tho]).astype(np.float32)


def thu_nho_torch(tho, khu_rang_cua):
    x = torch.from_numpy(tho).float().unsqueeze(1)
    y = F.interpolate(x, size=(28, 28), mode="bilinear", align_corners=False, antialias=khu_rang_cua)
    return y.squeeze(1).round().clamp(0, 255).numpy()


def chuan_hoa_mnist(x, tb=0.1307, dlc=0.3081, chia=255.0):
    return torch.from_numpy((x / chia - tb) / dlc).float().reshape(len(x), -1)


def bien_doi_huan_luyen(tho):
    """Hàm biến đổi của phía huấn luyện. Dùng chung hàm này ở phía phục vụ là cách phòng lệch đơn giản nhất."""
    return chuan_hoa_mnist(thu_nho_pil(tho))


X_hl = bien_doi_huan_luyen(tho_hl)
mlp = torch.nn.Sequential(torch.nn.Linear(784, 256), torch.nn.ReLU(), torch.nn.Linear(256, 10))
toi_uu = torch.optim.Adam(mlp.parameters(), 1e-3)
t0 = time.perf_counter()
for _ in range(3):
    thu_tu = torch.randperm(N_HL)
    for i in range(0, N_HL, 64):
        chon = thu_tu[i:i + 64]
        loss = F.cross_entropy(mlp(X_hl[chon]), y_hl[chon])
        toi_uu.zero_grad()
        loss.backward()
        toi_uu.step()
mlp.eval()
print(f"ảnh gốc {tho_hl.shape[1]}×{tho_hl.shape[2]} → đầu vào {X_hl.shape[1]} số; huấn luyện xong sau"
      f" {time.perf_counter() - t0:.1f} s (đo trên máy này)")

In [ ]:
X_khop = bien_doi_huan_luyen(tho_kt)
PHUC_VU = {
    "dùng chung hàm của huấn luyện": X_khop,
    "torch, song tuyến có khử răng cưa": chuan_hoa_mnist(thu_nho_torch(tho_kt, True)),
    "song tuyến không khử răng cưa (như OpenCV)": chuan_hoa_mnist(thu_nho_torch(tho_kt, False)),
    "láng giềng gần nhất": chuan_hoa_mnist(thu_nho_pil(tho_kt, Image.NEAREST)),
    "hằng số chuẩn hoá của ImageNet": chuan_hoa_mnist(thu_nho_pil(tho_kt), 0.485, 0.229),
    "quên chia cho 255": chuan_hoa_mnist(thu_nho_pil(tho_kt), chia=1.0),
}
NGUONG_DC = 2 / 255 / 0.3081      # giả định: kiểm thử đối chiếu chấp nhận lệch tối đa 2 mức xám


def do_chinh_xac(X, mo_hinh=mlp):
    with torch.no_grad():
        return 100 * (mo_hinh(X).argmax(1) == y_kt).float().mean().item()


ket_skew = {}
for ten, X in PHUC_VU.items():
    lech = (X - X_khop).abs().max().item()
    ket_skew[ten] = (do_chinh_xac(X), lech <= NGUONG_DC)
    do_tren_may(f"{ten}: độ chính xác", ket_skew[ten][0], "%")
    print(f"   lệch lớn nhất so với tensor của huấn luyện {lech:8.3f} → kiểm thử đối chiếu"
          f" {'đạt' if ket_skew[ten][1] else 'KHÔNG ĐẠT'}")
goc = ket_skew["dùng chung hàm của huấn luyện"][0]
print("\nxếp theo độ chính xác mất đi:", " > ".join(f"{k} ({goc - v[0]:.2f} điểm)" for k, v in
      sorted(ket_skew.items(), key=lambda kv: kv[1][0]) if k != "dùng chung hàm của huấn luyện"))
mot = tho_kt[0]
do_tren_may("thu nhỏ một ảnh bằng song tuyến của PIL", 1e6 * thoi_gian(lambda: Image.fromarray(mot).resize((28, 28), Image.BILINEAR), lap=500), "μs")
do_tren_may("thu nhỏ một ảnh bằng láng giềng gần nhất", 1e6 * thoi_gian(lambda: Image.fromarray(mot).resize((28, 28), Image.NEAREST), lap=500), "μs")
theo_sach("kịch bản minh hoạ của chương: mất (điểm phần trăm)", sach(95, nguon="95 · 90 %") - sach(90, nguon="95 · 90 %"),
          sach=5, nguon="90 % · 5 điểm")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.9))
ten_ds = list(ket_skew)
for y, ten in enumerate(ten_ds):
    acc, dat = ket_skew[ten]
    ax.plot([acc], [y], "o", color=MAU[0] if dat else MAU[1], ms=9, mec="white", mew=1.5)
    ax.text(acc - 0.6, y, f"{acc:.2f} % · {'đạt' if dat else 'không đạt'} đối chiếu", ha="right", va="center", fontsize=8)
ax.axvline(goc, color="0.6", lw=1, ls=":")
ax.set_yticks(range(len(ten_ds)), ten_ds)
ax.invert_yaxis()
ax.set_xlim(min(v[0] for v in ket_skew.values()) - 22, 100)
ax.set_xlabel("độ chính xác trên 5,000 ảnh kiểm tra (%)")
ax.set_title("Một mô hình, sáu cách tiền xử lý ở phía phục vụ (đo trên máy này)")
ax.plot([], [], "o", color=MAU[0], label="đạt kiểm thử đối chiếu")
ax.plot([], [], "o", color=MAU[1], label="không đạt")
ax.legend(loc="lower left", fontsize=8)
ax.grid(alpha=0.25, axis="x")
plt.tight_layout()
plt.show()

Trọng số giống hệt ở mọi dòng; chỉ hàm biến đổi đổi. Không dòng nào báo lỗi, và mọi dòng chạy với cùng độ
trễ, nên theo dõi độ trễ hay bắt ngoại lệ đều không thấy gì. Mức độ chính xác mất đi tuỳ vào loại lệch và
vào mô hình: có thay đổi làm mất nhiều điểm, có thay đổi gần như không. Con số 95 xuống 90 phần trăm của
chương là một kịch bản minh hoạ, và sách nói nó không phải một sự cố sản xuất được báo cáo.

Kiểm thử đối chiếu là cách bắt lệch trước khi nó tới người dùng. Nó so tensor của phía phục vụ với tensor
của phía huấn luyện trên cùng đầu vào, với một ngưỡng nhỏ. Nó không cần nhãn, và nó bắt cả những thay đổi
hôm nay chưa làm mất bao nhiêu độ chính xác, nhưng với một mô hình khác thì có thể. Dòng thứ hai cho thấy
điều ngược lại cũng đúng: một thư viện khác vẫn đạt đối chiếu nếu nó cài cùng thuật toán. Đường nhanh hơn
(láng giềng gần nhất) tiết kiệm được một chút tiền xử lý mỗi ảnh, nhưng điều cần đếm là số câu trả lời
**đúng** kịp hạn, không phải số câu trả lời kịp hạn.

## 7. Khởi động nguội

Phân tích hàng đợi giả định mô hình đã nạp và sẵn sàng. Trước yêu cầu đầu tiên, mô hình phải được nạp từ
nơi lưu trữ vào bộ nhớ và chuẩn bị cho suy luận. Bảng 8 là dòng thời gian minh hoạ của một lần khởi động
nguội cho ResNet-50. Sách ghi hai tổng, nhưng một vài dòng là khoảng, nên ta xem mỗi tổng đọc khoảng ấy ở
đâu.

In [ ]:
nap_ssd = sach(0.5, trich="Weight loading (SSD) | 0.5 s | 98 MB FP32 weights")
nap_s3 = (sach(3, trich="Weight loading (S3) | 3–5 s |"), sach(5, trich="Weight loading (S3) | 3–5 s |"))
cuda = (sach(0.3, trich="CUDA context | 0.3–0.5 s |"), sach(0.5, trich="CUDA context | 0.3–0.5 s |"))
bien_dich = (sach(15, trich="TensorRT compilation | 15–30 s |"), sach(30, trich="TensorRT compilation | 15–30 s |"))
lam_nong = sach(0.2, trich="Warmup (10 inferences) | 0.2 s |")
moi_truong = sach(0.4, trich="Runtime overhead | 0.4 s |")
KD_CLOUD = sach(35, trich="Total (cloud, first deploy) | 35 s |")


def giua(ab):
    return (ab[0] + ab[1]) / 2


cuc_bo = nap_ssd + giua(cuda) + lam_nong + moi_truong
print(f"cục bộ, engine đã biên dịch sẵn: {nap_ssd + cuda[0] + lam_nong + moi_truong:.1f}–"
      f"{nap_ssd + cuda[1] + lam_nong + moi_truong:.1f} s")
theo_sach("cục bộ, lấy giữa khoảng của CUDA context (s)", cuc_bo, sach=1.5, trich="Total (local, optimized) | 1.5 s |")
cloud_lo = nap_s3[0] + cuda[0] + bien_dich[0] + lam_nong + moi_truong
cloud_hi = nap_s3[1] + cuda[1] + bien_dich[1] + lam_nong + moi_truong
print(f"cloud, lần triển khai đầu: {cloud_lo:.1f}–{cloud_hi:.1f} s")
theo_sach("cloud, lấy giữa hai khoảng và 30 s biên dịch (s)", giua(nap_s3) + giua(cuda) + bien_dich[1] + lam_nong + moi_truong,
          sach=35, trich="Total (cloud, first deploy) | 35 s |")
print(f"bỏ pha biên dịch {bien_dich[1]} s nhờ lưu engine đã biên dịch: cloud còn {KD_CLOUD - bien_dich[1]:.1f} s")
print(f"mỗi lần suy luận làm nóng: {1000 * lam_nong / sach(10, trich='Warmup (10 inferences) | 0.2 s |'):.0f} ms")
TRONG_SO = sach(98, trich="Weight loading (SSD) | 0.5 s | 98 MB FP32 weights")
NVME = sach(500, nguon="50 GB · 500 MB/s")
print(f"{TRONG_SO} MB trong {nap_ssd} s là {TRONG_SO / nap_ssd:.0f} MB/s; ổ NVMe {NVME} MB/s đọc chừng ấy trong"
      f" {TRONG_SO / NVME:.2f} s")
print()
MH_10, PCIE = sach(10, nguon="10 GB · 32 GB/s"), sach(32, nguon="10 GB · 32 GB/s")
theo_sach("hoán đổi mô hình 10 GB qua PCIe Gen4 x16 (ms)", 1000 * MH_10 / PCIE, sach=312.5, nguon="32 GB/s · 312.5 ms")
print(f"gấp {1000 * MH_10 / PCIE / sach(50, nguon='312.5 ms · 50 ms'):.2f} lần SLO 50 ms, trước cả giải tuần tự hóa và làm nóng")
CK = sach(5, nguon="1.12 · 5 GB")
pickle_s, st_s = sach(15, nguon="5 GB · 15 giây"), sach(1.5, nguon="15 giây · 1.5 giây")
theo_sach("ví dụ 1.12: Safetensors nhanh hơn torch.load (lần)", pickle_s / st_s, sach=10, nguon="1.5 giây · 10 lần")
print(f"tức {1000 * CK / pickle_s:.0f} MB/s so với {CK / st_s:.2f} GB/s cho checkpoint {CK} GB")

Giờ đo trên máy này. Ta ghi trọng số của ResNet-50 ra một tệp (khoảng 100 MB, trong thư mục làm việc,
xoá ở cuối ô), rồi nạp lại theo ba cách: `torch.load` thường (đọc và dựng lại mọi tensor), `torch.load` với
`mmap=True` (ánh xạ bộ nhớ: chỉ gắn tệp vào không gian địa chỉ), và lần đầu chạm vào mọi trọng số sau khi
ánh xạ.

Một tệp vừa ghi còn nằm trong bộ nhớ đệm trang của hệ điều hành, và đọc từ đó là chép bộ nhớ chứ không
phải đọc đĩa. Trên Linux, `os.posix_fadvise(..., POSIX_FADV_DONTNEED)` xin hệ điều hành bỏ các trang của
tệp ấy khỏi bộ nhớ đệm, nên lần đọc sau thật sự đi xuống đĩa. Đây chỉ là lời xin: ô sau in ra hệ điều hành
có làm theo không.

**Dự đoán:** ánh xạ bộ nhớ mở tệp gần như tức thì. Chi phí đọc đĩa có biến mất không, hay chỉ dời đi đâu?

In [ ]:
TEP = "resnet50_thu.pt"
torch.save(resnet.state_dict(), TEP)
xin_duoc = hasattr(os, "posix_fadvise")


def bo_khoi_bo_dem_trang(duong_dan):
    gc.collect()
    if xin_duoc:
        fd = os.open(duong_dan, os.O_RDONLY)
        os.fsync(fd)
        os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
        os.close(fd)


def cham_het(sd):
    return sum(float(t.float().sum()) for t in sd.values())


do = {k: [] for k in ("torch.load, tệp nguội", "torch.load, tệp nóng", "mmap: mở tệp nguội",
                      "mmap: lần đầu chạm mọi trọng số", "mmap: lần chạm thứ hai")}
for _ in range(3):
    bo_khoi_bo_dem_trang(TEP)
    t0 = time.perf_counter()
    sd = torch.load(TEP, weights_only=True)
    do["torch.load, tệp nguội"].append(time.perf_counter() - t0)
    del sd
    t0 = time.perf_counter()
    sd = torch.load(TEP, weights_only=True)
    do["torch.load, tệp nóng"].append(time.perf_counter() - t0)
    del sd
    bo_khoi_bo_dem_trang(TEP)
    t0 = time.perf_counter()
    sd = torch.load(TEP, weights_only=True, mmap=True)
    t1 = time.perf_counter()
    cham_het(sd)
    t2 = time.perf_counter()
    cham_het(sd)
    t3 = time.perf_counter()
    do["mmap: mở tệp nguội"].append(t1 - t0)
    do["mmap: lần đầu chạm mọi trọng số"].append(t2 - t1)
    do["mmap: lần chạm thứ hai"].append(t3 - t2)
    del sd
co_tep = os.path.getsize(TEP)
print(f"tệp {co_tep / 1e6:.1f} MB")
for ten, ts in do.items():
    do_tren_may(ten, 1000 * float(np.median(ts)), "ms")
ty = np.median(do["torch.load, tệp nguội"]) / np.median(do["torch.load, tệp nóng"])
print(f"đọc nguội chậm hơn đọc nóng {ty:.1f} lần" + ("" if ty > 2 else
      "; hệ điều hành có vẻ không bỏ trang khỏi bộ nhớ đệm, nên các số 'nguội' ở đây không đáng tin"))

# Yêu cầu đầu tiên của một bản sao vừa nạp bằng mmap, so với yêu cầu thứ hai
t0 = time.perf_counter()
moi = torchvision.models.resnet50(weights=None)
t_dung = time.perf_counter() - t0
t0 = time.perf_counter()
with torch.device("meta"):
    moi = torchvision.models.resnet50()
t_meta = time.perf_counter() - t0
bo_khoi_bo_dem_trang(TEP)
t0 = time.perf_counter()
moi.load_state_dict(torch.load(TEP, weights_only=True, mmap=True), assign=True)
moi.eval()
t_gan = time.perf_counter() - t0
with torch.inference_mode():
    t0 = time.perf_counter()
    moi(x)
    t_dau = time.perf_counter() - t0
    t0 = time.perf_counter()
    moi(x)
    t_hai = time.perf_counter() - t0
do_tren_may("dựng ResNet-50 có khởi tạo ngẫu nhiên", 1000 * t_dung, "ms")
do_tren_may("dựng trên thiết bị meta (không khởi tạo)", 1000 * t_meta, "ms")
do_tren_may("gắn trọng số ánh xạ từ tệp nguội", 1000 * t_gan, "ms")
do_tren_may("yêu cầu đầu tiên", 1000 * t_dau, "ms")
do_tren_may("yêu cầu thứ hai", 1000 * t_hai, "ms")
del moi
os.remove(TEP)
print(f"đã xoá {TEP}")

Ánh xạ bộ nhớ không xoá chi phí đọc, mà dời nó đi: tệp mở gần như tức thì, rồi lần đầu chạm vào trọng số
mới đọc các trang từ đĩa. Với một mô hình dày đặc, lượt truyền xuôi đầu tiên chạm gần hết trọng số, nên
yêu cầu đầu tiên trả phần ấy. Đó là điều chương nói về ánh xạ bộ nhớ: khởi động có vẻ ngắn hơn, còn độ trễ
nạp bị dời sang những yêu cầu đầu tiên, và dịch vụ nhạy độ trễ vẫn nạp trước hay làm nóng. Dựng mô hình
với khởi tạo ngẫu nhiên rồi mới ghi đè trọng số cũng là việc thừa; dựng trên thiết bị `meta` bỏ được nó.

### Một đợt tăng vọt khi bản sao mới khởi động nguội

Ngộ nhận thứ tư của chương cho rằng khởi động nguội chỉ quan trọng với yêu cầu đầu tiên. Thật ra nó cộng
dồn khi lưu lượng tăng vọt. Ta mô phỏng một dịch vụ (mọi thông số là **giả định của sổ tay**, trừ những số
ghi nguồn): 4 bản sao, mỗi bản sao phục vụ trung bình 5 ms một yêu cầu, lưu lượng nền 500 yêu cầu mỗi giây.
Ở giây thứ 20, lưu lượng lên 1,500 yêu cầu mỗi giây trong 60 giây. Bộ tự động điều chỉnh quy mô nhận ra sau
5 giây và xin thêm 8 bản sao. Bốn chính sách khác nhau ở lúc 8 bản sao ấy dùng được.

**Dự đoán:** trong đợt tăng, bao nhiêu phần trăm yêu cầu kịp hạn 50 ms nếu bản sao mới cần 35 giây?

In [ ]:
S_BS = sach(5, nguon="500 ms · 5 ms") / 1000    # thời gian phục vụ trung bình
NGUOI = sach(500, nguon="300 giây · 500 ms") / 1000   # một yêu cầu nguội khi chưa làm nóng
SLO_TV = sach(50, nguon="50 ms · 5,000 QPS") / 1000
BAN_DAU, THEM, PHAT_HIEN = 4, 8, 5.0
DOAN = ((500, 0, 20), (1500, 20, 60), (500, 80, 40))     # (yêu cầu mỗi giây, bắt đầu, kéo dài)
rng = np.random.default_rng(42)
den = np.concatenate([np.sort(rng.uniform(a, a + d, rng.poisson(lam * d))) for lam, a, d in DOAN])


def phuc_vu(den, san_sang, rng, nguoi=None):
    """FCFS qua các bản sao; bản sao i dùng được từ san_sang[i]. nguoi: thời gian yêu cầu đầu của bản sao mới."""
    t_pv = rng.exponential(S_BS, len(den))
    dong = [(t, i) for i, t in enumerate(san_sang)]
    heapq.heapify(dong)
    da_phuc_vu = [0] * len(san_sang)
    tre = np.empty(len(den))
    for k, a in enumerate(den):
        ranh, i = heapq.heappop(dong)
        bat_dau = max(a, ranh)
        s = nguoi if (nguoi is not None and san_sang[i] > 0 and da_phuc_vu[i] == 0) else t_pv[k]
        da_phuc_vu[i] += 1
        tre[k] = bat_dau + s - a
        heapq.heappush(dong, (bat_dau + s, i))
    return tre


CHINH_SACH = {
    f"khởi động nguội {KD_CLOUD} s (cloud, lần đầu)": (KD_CLOUD, None),
    f"engine biên dịch sẵn, {cuc_bo} s": (cuc_bo, None),
    f"{cuc_bo} s, không làm nóng: yêu cầu đầu {1000 * NGUOI:.0f} ms": (cuc_bo, NGUOI),
    "nhóm bản sao chờ sẵn": (0.0, None),
}
dot = (den >= 20) & (den < 80)
ket_tv = {}
for ten, (tre_kd, nguoi) in CHINH_SACH.items():
    san_sang = [0.0] * BAN_DAU + [20 + PHAT_HIEN + tre_kd] * THEM
    ket_tv[ten] = phuc_vu(den, san_sang, np.random.default_rng(7), nguoi)
    t = ket_tv[ten]
    print(f"{ten:<46} kịp hạn trong đợt tăng {np.mean(t[dot] <= SLO_TV):6.1%} · cả 120 s {np.mean(t <= SLO_TV):6.1%}"
          f" · lâu nhất {t.max():5.1f} s")
print()
theo_sach("10 thực thể × 30 giây biên dịch (giây theo thực thể)", sach(10, nguon="10 thực thể") * sach(30, nguon="10 thực thể · 30 giây"),
          sach=300, nguon="30 giây · 300 giây")
print(f"yêu cầu nguội so với ổn định: {NGUOI / S_BS:.0f} lần")

In [ ]:
O = 1.0
bien = np.arange(0, 120 + O, O)
o = np.digitize(den, bien) - 1
giua_o = bien[:-1] + O / 2
fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.axvspan(20, 80, color="0.93", lw=0)
for (ten, t), mau in zip(ket_tv.items(), (MAU[1], MAU[0], MAU[3], MAU[2])):
    p99 = [1000 * np.percentile(t[o == k], 99) if np.any(o == k) else np.nan for k in range(len(giua_o))]
    ax.plot(giua_o, p99, color=mau, lw=2, label=ten)
ax.axhline(1000 * SLO_TV, color="0.3", lw=1, ls="--")
ax.text(1, 1000 * SLO_TV * 1.25, "hạn 50 ms", fontsize=8, color="0.3")
ax.text(50, 2, "đợt tăng: 1,500 yêu cầu mỗi giây", ha="center", fontsize=8, color="0.3")
ax.set_yscale("log")
ax.set_ylim(1, 1e5)
ax.set_xlim(0, 120)
ax.set_xlabel("thời gian (s)")
ax.set_ylabel("p99 mỗi giây (ms, thang log)")
ax.set_title("Đợt tăng vọt và lúc bản sao mới dùng được (giả định của sổ tay, mô phỏng)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

Trong lúc bản sao mới chưa dùng được, tốc độ đến vượt năng lực và phần tồn đọng tích lại; nó còn phải tiêu
hết sau khi năng lực đã đủ, nên đợt tăng làm hỏng độ trễ lâu hơn chính nó. Với 35 giây khởi động nguội,
gần như mọi yêu cầu trong đợt tăng trễ hạn. Engine biên dịch sẵn rút khoảng chờ xuống gần bằng thời gian
phát hiện, còn nhóm bản sao chờ sẵn chỉ phải trả đúng thời gian phát hiện. Bỏ làm nóng thêm một ít độ trễ
đuôi: mỗi bản sao mới phục vụ yêu cầu đầu tiên chậm gấp trăm lần. Đó là lý do chương nói chính sách chỉ
đếm bản sao ở trạng thái ổn định sẽ đánh giá thấp năng lực cần có, và ngân sách mở rộng phải gồm biên dịch,
nạp trọng số, khởi tạo bộ đệm và kiểm tra sức khoẻ.

## 8. Môi trường chạy và độ chính xác số

Bảng 19 là một kịch bản minh hoạ cho ResNet-50 batch 1 trên V100, với độ trễ giả định và mức tăng tốc suy
ra. Cột tăng tốc là mốc PyTorch tức thời chia cho từng độ trễ.

In [ ]:
moc19 = sach(8.5, nguon="50 · 8.5 ms")    # PyTorch tức thời, mốc cơ sở
BANG19 = {"TorchScript": sach(6.2, nguon="8.5 ms · 6.2 ms"), "ONNX Runtime": sach(5.1, nguon="1.4 lần · 5.1 ms"),
          "TensorRT FP32": sach(2.8, nguon="1.7 lần · 2.8 ms"), "TensorRT FP16": sach(1.4, nguon="3 lần · 1.4 ms"),
          "TensorRT INT8": sach(0.9, nguon="6.1 lần · 0.9 ms")}
theo_sach("TorchScript: tăng tốc", moc19 / BANG19["TorchScript"], sach=1.4, nguon="6.2 ms · 1.4 lần")
theo_sach("ONNX Runtime: tăng tốc", moc19 / BANG19["ONNX Runtime"], sach=1.7, nguon="5.1 ms · 1.7 lần")
theo_sach("TensorRT FP32: tăng tốc", moc19 / BANG19["TensorRT FP32"], sach=3, nguon="2.8 ms · 3 lần")
theo_sach("TensorRT FP16: tăng tốc", moc19 / BANG19["TensorRT FP16"], sach=6.1, nguon="1.4 ms · 6.1 lần")
theo_sach("TensorRT INT8: tăng tốc", moc19 / BANG19["TensorRT INT8"], sach=9.4, nguon="0.9 ms · 9.4 lần")
print(f"từ FP32 sang FP16 trên TensorRT: nhanh {BANG19['TensorRT FP32'] / BANG19['TensorRT FP16']:.1f} lần; chỉ đổi môi trường"
      f" chạy, vẫn FP32: nhanh {moc19 / BANG19['TensorRT FP32']:.1f} lần")

Kỹ thuật đầu tiên của các engine chuyên biệt là hợp nhất các tầng liên tiếp. Chương xét tích chập, rồi chuẩn
hóa theo batch, rồi ReLU: không hợp nhất thì cần ba lần khởi chạy kernel và hai vòng ghi rồi đọc lại tensor
trung gian. Lúc suy luận, chuẩn hóa theo batch chỉ là một phép nhân và một phép cộng theo kênh với hằng số
đã biết, nên nó gộp hẳn được vào trọng số của tích chập trước nó. Đó cũng là tính sẵn hằng số (constant
folding): việc làm một lần lúc triển khai thay cho ở mọi yêu cầu. Ta gộp cả 53 lớp chuẩn hóa theo batch
của ResNet-50 rồi đo trên máy này. Để phép thử có nghĩa, các thống kê của chuẩn hóa theo batch được gán
ngẫu nhiên có hạt giống trước (giả định của sổ tay), vì giá trị mặc định của chúng gần như không làm gì.

**Dự đoán:** đầu ra sau khi gộp có giống hệt từng bit không?

In [ ]:
dat_hat_giong()
goc_bn = copy.deepcopy(resnet)
sinh = torch.Generator().manual_seed(42)
for m in goc_bn.modules():
    if isinstance(m, torch.nn.BatchNorm2d):
        n = m.num_features
        m.weight.data = 0.5 + torch.rand(n, generator=sinh)
        m.bias.data = 0.1 * torch.randn(n, generator=sinh)
        m.running_mean = 0.1 * torch.randn(n, generator=sinh)
        m.running_var = 0.5 + 1.5 * torch.rand(n, generator=sinh)


def gop_bn(mo_hinh):
    """Gộp mỗi BatchNorm2d vào tích chập ngay trước nó: w' = w·γ/σ, b' = β − γ·μ/σ."""
    m = copy.deepcopy(mo_hinh)

    def gop(conv, bn):
        he_so = bn.weight / torch.sqrt(bn.running_var + bn.eps)
        conv.weight.data = conv.weight.data * he_so.view(-1, 1, 1, 1)
        conv.bias = torch.nn.Parameter(bn.bias - bn.running_mean * he_so)

    gop(m.conv1, m.bn1)
    m.bn1 = torch.nn.Identity()
    for tang in (m.layer1, m.layer2, m.layer3, m.layer4):
        for khoi in tang:
            for i in (1, 2, 3):
                gop(getattr(khoi, f"conv{i}"), getattr(khoi, f"bn{i}"))
                setattr(khoi, f"bn{i}", torch.nn.Identity())
            if khoi.downsample is not None:
                gop(khoi.downsample[0], khoi.downsample[1])
                khoi.downsample[1] = torch.nn.Identity()
    return m.eval()


da_gop = gop_bn(goc_bn)
so_bn = sum(isinstance(m, torch.nn.BatchNorm2d) for m in goc_bn.modules())
byte_bn = [0]


def dem_byte(mod, vao, ra):
    byte_bn[0] += ra.numel() * ra.element_size()


moc = [m.register_forward_hook(dem_byte) for m in goc_bn.modules() if isinstance(m, torch.nn.BatchNorm2d)]
with torch.no_grad():
    y_goc = goc_bn(x)
for m in moc:
    m.remove()
with torch.inference_mode():
    y_gop = da_gop(x)
    t_goc = thoi_gian(lambda: goc_bn(x), lap=5, khoi_dong=1)
    t_gop = thoi_gian(lambda: da_gop(x), lap=5, khoi_dong=1)
lech = (y_goc - y_gop).abs().max().item() / y_goc.abs().max().item()
print(f"{so_bn} lớp chuẩn hóa theo batch được gộp: mỗi ảnh bớt {so_bn} lần khởi chạy kernel, và khoảng"
      f" {2 * byte_bn[0] / 1e6:.0f} MB tensor trung gian phải ghi ra rồi đọc lại (ước lượng: hai tensor cỡ đầu ra"
      f" của mỗi lớp)")
print(f"đầu ra: giống hệt từng bit? {'có' if torch.equal(y_goc, y_gop) else 'không'}; lệch tương đối lớn nhất {lech:.1e}")
do_tren_may("ResNet-50 chưa gộp", 1000 * t_goc, "ms")
do_tren_may("ResNet-50 đã gộp chuẩn hóa theo batch", 1000 * t_gop, "ms")
do_tren_may("nhanh lên", t_goc / t_gop, "lần")
del goc_bn, da_gop

Phép gộp đúng về toán học nhưng không đúng từng bit: nhân hệ số vào trọng số trước hay nhân vào đầu ra sau
cho làm tròn khác nhau. Đó là một dạng nhỏ của điều chương nói, tối ưu càng mạnh thì kết quả số càng có thể
lệch khỏi lúc huấn luyện. Trên CPU, phần lớn thời gian vẫn là tích chập, nên mức lợi ở đây thường khiêm tốn; trên
GPU, nơi một kernel chuẩn hóa theo batch nghẽn ở băng thông, bớt những vòng ghi và đọc ấy có giá trị hơn.

Bảng 20 là các giả định minh hoạ về độ chính xác số trên V100, và phương trình 16 cho trần chỉ tính băng
thông khi hạ độ chính xác.

In [ ]:
fp32_ms, int8_ms = sach(2.8, nguon="20 · 2.8 ms"), sach(0.9, nguon="85 % · 0.9 ms")
theo_sach("INT8 so với FP32: tăng tốc", fp32_ms / int8_ms, sach=3.1, nguon="76.05 % · 3.1 lần")
acc32, acc_ptq, acc_qat = sach(76.13, nguon="98 MB · 76.13 %"), sach(75.80, nguon="25 MB · 75.80 %"), sach(76.05, nguon="1,000 mẫu · 76.05 %")
theo_sach("PTQ mất (điểm phần trăm)", acc32 - acc_ptq, sach=0.33, nguon="3.1 lần · 0.33 điểm")
print(f"QAT lấy lại {100 * (acc_qat - acc_ptq) / (acc32 - acc_ptq):.0f} % phần mất ấy, còn mất {acc32 - acc_qat:.2f} điểm")
mb32 = sach(98, nguon="2.8 ms · 98 MB")
theo_sach("FP16: bộ nhớ (MB)", mb32 / 2, sach=49, nguon="1.4 ms · 49 MB")
theo_sach("INT8: bộ nhớ (MB)", mb32 / 4, sach=25, nguon="0.9 ms · 25 MB")
print(f"ResNet-50 có {SO_THAM_SO:,} tham số: FP32 là {4 * SO_THAM_SO / 1e6:.1f} MB thập phân, {4 * SO_THAM_SO / 2**20:.1f} MiB,"
      f" nên 98 MB của bảng là MiB")
theo_sach("phương trình 16: trần của INT8 so với FP32", sach(32, trich=r"= \frac{32}{8} = 4\times") / sach(8, trich=r"= \frac{32}{8} = 4\times"),
          sach=4, trich=r"= \frac{32}{8} = 4\times")
print(f"kịch bản trên V100 giả định thông lượng gấp ba, thấp hơn trần ấy; INT8 trong bảng 20 nhanh gấp {fp32_ms / int8_ms:.1f} lần")
GAP_BA = 3   # "gấp ba": chương viết bằng chữ
theo_sach("tải cần 30 GPU ở FP32, ở INT8 thông lượng gấp ba", sach(30, nguon="30 GPU · 10 GPU") / GAP_BA, sach=10,
          nguon="30 GPU · 10 GPU")

Phương trình 17 cho bước của một bộ lượng tử đều b bit trên khoảng đã hiệu chuẩn từ α tới β: Δ = (β − α) /
(2^b − 1). Giá trị trong khoảng sai tối đa nửa bước; giá trị ngoài khoảng bị cắt và có thể sai nhiều hơn.
Lượng tử hóa sau huấn luyện lấy α và β từ tập hiệu chuẩn, nên tập ấy phải đại diện cho lưu lượng thật.

Ta lượng tử hóa mạng MNIST của phần 6 ở INT8: trọng số đối xứng theo tầng, và đầu vào của mỗi tầng tuyến
tính theo khoảng [min, max] của một tập hiệu chuẩn 512 ảnh. Ba tập hiệu chuẩn: một tập đại diện, một tập
chỉ có chữ số 1, và một tập chụp tối hơn một nửa (giả định của sổ tay: như ảnh từ một camera khác).

**Dự đoán:** tập nào cho khoảng hẹp nhất ở tầng đầu, và điều gì xảy ra với những điểm ảnh sáng hơn khoảng ấy?

In [ ]:
def luong_tu(x, lo, hi, bit=8):
    """Phương trình 17: lượng tử đều trên [lo, hi], cắt phần ngoài khoảng, rồi giải lượng tử."""
    buoc = (hi - lo) / (2 ** bit - 1)
    return torch.clamp(torch.round((x - lo) / buoc), 0, 2 ** bit - 1) * buoc + lo, buoc


def trong_so_int8(w):
    s = w.abs().max() / 127
    return torch.clamp(torch.round(w / s), -127, 127) * s


def khoang_vao(X):
    ra, h = [], X
    with torch.no_grad():
        for l in mlp:
            if isinstance(l, torch.nn.Linear):
                ra.append((h.min().item(), h.max().item()))
            h = l(h)
    return ra


def chay_int8(X, khoang):
    h, k = X, 0
    with torch.no_grad():
        for l in mlp:
            if isinstance(l, torch.nn.Linear):
                h = F.linear(luong_tu(h, *khoang[k])[0], trong_so_int8(l.weight), l.bias)
                k += 1
            else:
                h = l(h)
    return h


chon = torch.randperm(N_HL, generator=torch.Generator().manual_seed(42))[:512]
TAP_HC = {"đại diện: 512 ảnh ngẫu nhiên": X_hl[chon],
          "chỉ chữ số 1": X_hl[y_hl == 1][:512],
          "ảnh tối hơn một nửa": bien_doi_huan_luyen((tho_hl[chon.numpy()] * 0.5).round().astype(np.uint8))}
do_tren_may("FP32, không lượng tử hóa: độ chính xác", do_chinh_xac(X_khop), "%")
for ten, tap in TAP_HC.items():
    kh = khoang_vao(tap)
    lo, hi = kh[0]
    xq, buoc = luong_tu(X_khop, lo, hi)
    trong = (X_khop >= lo) & (X_khop <= hi)
    print(f"{ten}:\n   tầng 1 nhận [{lo:.2f}, {hi:.2f}], bước Δ = {buoc:.4f}; {100 * (~trong).float().mean():.2f} % giá trị"
          f" kiểm tra bị cắt; sai lớn nhất trong khoảng {(xq - X_khop)[trong].abs().max():.4f} (Δ/2 = {buoc / 2:.4f}),"
          f" ngoài khoảng {(xq - X_khop)[~trong].abs().max() if (~trong).any() else 0:.3f}")
    print(f"   tầng 2 nhận [{kh[1][0]:.2f}, {kh[1][1]:.2f}]; một mức xám sau chuẩn hoá là {1 / 255 / 0.3081:.4f}")
    with torch.no_grad():
        acc = 100 * (chay_int8(X_khop, kh).argmax(1) == y_kt).float().mean().item()
    do_tren_may(f"INT8 hiệu chuẩn bằng tập {ten}: độ chính xác", acc, "%")
theo_sach("kịch bản minh hoạ của chương: mất (điểm phần trăm)", sach(76.1, nguon="25 ms · 76.1") - sach(72.9, nguon="76.1 · 72.9 %"),
          sach=3.2, nguon="72.9 % · 3.2 điểm")

Với tập đại diện, khoảng ở tầng đầu đúng bằng khoảng của ảnh 8 bit sau chuẩn hoá, nên bước Δ trùng đúng một
mức xám và tầng đầu không sai chút nào: đầu vào vốn đã chỉ có 256 giá trị. Tập tối hơn cho khoảng hẹp hơn
một nửa. Trong khoảng, sai số không vượt Δ/2, đúng như phương trình 17 nói, nhưng mọi nét chữ sáng hơn khoảng
ấy bị cắt, và đó là chỗ sai lớn. Tập chữ số 1 vẫn chứa điểm ảnh sáng nhất, nên khoảng ở tầng đầu không đổi;
nó chỉ khác ở tầng ẩn. Độ chính xác mất bao nhiêu tuỳ vào mô hình và vào chỗ bị cắt, và mạng nhỏ này chịu
đựng khá tốt ở 8 bit. Sách kết luận đúng điều ấy: sai số ở mức tầng hay mức tác vụ phải đo chứ không suy ra
được, và hiệu chuẩn cần kiểm định trên dữ liệu đại diện cho đầu vào mà dịch vụ sẽ gặp.

## 9. Hoá đơn: chi phí mỗi lần suy luận

Chi phí mỗi lần suy luận chia chi phí phục vụ được phân bổ cho số lần suy luận hoàn thành mà vẫn đạt mục
tiêu dịch vụ. Ví dụ 1.3 của chương xét một máy chủ GPU thuê 4 đô la mỗi giờ, chạy batch 1 hoặc batch 8.
Bảng 22 là một kịch bản giá theo giờ kiểu AWS cho ResNet-50, với thông lượng bão hoà giả định.

**Dự đoán:** máy nào rẻ nhất cho một triệu ảnh: CPU, T4 hay V100?

In [ ]:
def moi_trieu(gia_gio, moi_giay):
    return gia_gio / (moi_giay * 3600) * 1e6


GIA_13 = sach(4, nguon="1.3 · 4 đô la")
b1, b8 = sach(1, trich="Batch size 1. Latency: 5 ms."), sach(8, trich="Batch size 8. Latency: 10 ms")
tl_a = b1 / (sach(5, nguon="4 đô la · 5 ms") / 1000)
tl_b = b8 / (sach(10, nguon="5 ms · 10 ms") / 1000)
theo_sach("kịch bản A: thông lượng (yêu cầu/s)", tl_a, sach=200, nguon="10 ms · 200 yêu cầu")
theo_sach("kịch bản B: thông lượng (yêu cầu/s)", tl_b, sach=800, nguon="200 yêu cầu · 800")
theo_sach("A: đô la mỗi triệu truy vấn", moi_trieu(GIA_13, tl_a), sach=5.56, nguon="800 · 5.56 đô la")
theo_sach("B: đô la mỗi triệu truy vấn", moi_trieu(GIA_13, tl_b), sach=1.39, nguon="5.56 đô la · 1.39 đô la")
theo_sach("hạ độ trễ từ 10 xuống 5 ms làm hoá đơn tăng (%)", 100 * (moi_trieu(GIA_13, tl_a) / moi_trieu(GIA_13, tl_b) - 1),
          sach=300, nguon="1.39 đô la · 300 %")
print()
MAY = ("CPU (c5.xlarge)", "T4 (g4dn.xlarge)", "V100 (p3.2xlarge)")   # bảng 22
GIA = dict(zip(MAY, (sach(0.17, nguon="50 · 0.17 đô la"), sach(0.53, nguon="0.94 đô la · 0.53 đô la"),
                     sach(3.06, nguon="0.37 đô la · 3.06 đô la"))))          # đô la mỗi giờ
TL = dict(zip(MAY, (sach(50, nguon="0.17 đô la · 50 ảnh"), sach(400, nguon="0.53 đô la · 400 ảnh"),
                    sach(1200, nguon="3.06 đô la · 1,200 ảnh"))))            # ảnh mỗi giây, bão hoà
theo_sach("CPU: đô la mỗi triệu ảnh", moi_trieu(GIA[MAY[0]], TL[MAY[0]]), sach=0.94, nguon="50 ảnh · 0.94 đô la")
theo_sach("T4: đô la mỗi triệu ảnh", moi_trieu(GIA[MAY[1]], TL[MAY[1]]), sach=0.37, nguon="400 ảnh · 0.37 đô la")
theo_sach("V100: đô la mỗi triệu ảnh", moi_trieu(GIA[MAY[2]], TL[MAY[2]]), sach=0.71, nguon="1,200 ảnh · 0.71 đô la")
print(f"V100 có thông lượng gấp {TL['V100 (p3.2xlarge)'] / TL['T4 (g4dn.xlarge)']:.0f} lần T4, nhưng giá theo giờ gấp"
      f" {GIA['V100 (p3.2xlarge)'] / GIA['T4 (g4dn.xlarge)']:.1f} lần")

Chương mở mục này bằng một câu hỏi: một nhóm đạt 1,200 ảnh mỗi giây trên V100, vậy với mục tiêu 5,000 QPS
và cùng một SLO thì 8 chiếc V100 giá 3 đô la mỗi giờ hay 24 chiếc T4 giá 0.53 đô la mỗi giờ rẻ hơn? Chương
không nói 8 và 24 từ đâu ra. Một cách tái dựng khớp cả hai: mỗi máy chạy ở ngưỡng mức sử dụng an toàn 54 %
của ví dụ hoạch định năng lực (sổ tay 01 tính lại ngưỡng ấy cho SLO p99 50 ms). Để ý V100 ở đây giá 3 đô
la, còn trong bảng 22 là 3.06.

In [ ]:
QPS = sach(5000, nguon="1,200 ảnh · 5,000 QPS")
RHO = sach(0.54, nguon="4.6 · 0.54 · 54 %")
tl_v100 = sach(1200, nguon="1,200 ảnh · 5,000 QPS")
so_v100 = math.ceil(QPS / (RHO * tl_v100))
so_t4 = math.ceil(QPS / (RHO * TL["T4 (g4dn.xlarge)"]))
theo_sach("số V100 ở mức sử dụng 54 %", so_v100, sach=8, trich="whether 8 V100s at $3/hour each or 24 T4s")
theo_sach("số T4 ở mức sử dụng 54 %", so_t4, sach=24, trich="whether 8 V100s at $3/hour each or 24 T4s")
gia_v100 = sach(3, nguon="5,000 QPS · 3 đô la")
gia_t4 = sach(0.53, nguon="24 · 0.53 đô la")
GIO_MOT_NAM = 24 * 365
print(f"{so_v100} V100: {so_v100 * gia_v100:.2f} đô la mỗi giờ · {so_t4} T4: {so_t4 * gia_t4:.2f} đô la mỗi giờ"
      f" → T4 rẻ hơn {so_v100 * gia_v100 / (so_t4 * gia_t4):.2f} lần, tức"
      f" {GIO_MOT_NAM * (so_v100 * gia_v100 - so_t4 * gia_t4):,.0f} đô la một năm")
print(f"ở mức sử dụng thật: V100 {100 * QPS / (so_v100 * tl_v100):.0f} %, T4 {100 * QPS / (so_t4 * TL['T4 (g4dn.xlarge)']):.0f} %")

Bảng 22 tính ở thông lượng bão hoà, tức mức sử dụng 100 %. Ở mức sử dụng thấp, năng lực đã trả tiền bị bỏ
phí, nên cùng hoá đơn theo giờ bị chia cho ít lần suy luận hơn. Và một đội máy chỉ có số nguyên máy: với lưu
lượng nhỏ, một chiếc T4 phần lớn thời gian nhàn rỗi. Hình sau vẽ cả hai điều ấy, với cùng ngưỡng 54 % cho
mọi loại máy (giả định của sổ tay).

In [ ]:
qps = np.logspace(0, 4.2, 600)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3))
gio = {}
for (ten, mau) in zip(MAY, (MAU[2], MAU[0], MAU[1])):
    so_may = np.ceil(qps / (RHO * TL[ten]))
    gio[ten] = so_may * GIA[ten]
    ax1.step(qps, gio[ten], where="post", color=mau, lw=2, label=ten)
    u = np.linspace(0.05, 1, 200)
    ax2.plot(100 * u, moi_trieu(GIA[ten], u * TL[ten]), color=mau, lw=2, label=ten)
re_nhat = np.array([min(gio, key=lambda k: gio[k][i]) for i in range(len(qps))])
doi = np.flatnonzero(re_nhat[1:] != re_nhat[:-1])
for i in doi:
    print(f"từ khoảng {qps[i + 1]:,.0f} QPS, loại máy rẻ nhất đổi từ {re_nhat[i]} sang {re_nhat[i + 1]}")
ax1.axvline(QPS, color="0.6", lw=1, ls=":")
ax1.text(QPS * 0.92, 0.25, "5,000 QPS", ha="right", fontsize=8, color="0.3")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("lưu lượng cần phục vụ (QPS, thang log)")
ax1.set_ylabel("đô la mỗi giờ của cả đội máy (thang log)")
ax1.set_title("Chi phí đội máy ở ngưỡng 54 % (giả định)")
ax1.legend(loc="upper left", fontsize=8)
ax1.grid(alpha=0.25, which="both")
ax2.set_yscale("log")
ax2.set_xlim(0, 100)
ax2.set_xlabel("mức sử dụng (%)")
ax2.set_ylabel("đô la mỗi triệu ảnh (thang log)")
ax2.set_title("Chi phí mỗi triệu ảnh theo mức sử dụng (bảng 22)")
ax2.legend(loc="upper right", fontsize=8)
ax2.grid(alpha=0.25, which="both")
plt.tight_layout()
plt.show()

Bên phải: chi phí mỗi ảnh tỉ lệ nghịch với mức sử dụng, nên một máy GPU chỉ rẻ khi được giữ bận. Bên trái:
ở lưu lượng thấp, máy CPU rẻ nhất vì một máy GPU nhỏ nhất đã thừa năng lực; khi lưu lượng lớn, T4 thắng.
Đó là điều chương nói về GPU và CPU: suy luận trên CPU hợp với lưu lượng thấp hay biến động, còn GPU hợp
với lưu lượng cao và đều đủ giữ mức sử dụng. Với những giả định này, V100 không bao giờ là máy rẻ nhất;
nó chỉ đáng khi một yêu cầu cần độ trễ mà T4 không đạt được. Sổ tay 01 đi trọn quy trình hoạch định năng
lực tới 62 GPU V100 cho ví dụ của chương.

## Thử thêm

1. Lấy một chuỗi mã token thay cho embedding: `ma = np.random.default_rng(42).integers(0, 50_000, 1000)`.
   JSON của số nguyên ngắn hơn nhiều so với số thực (`len(json.dumps(ma.tolist()))`). Protobuf mã hoá
   `repeated int32` bằng varint, nên mỗi mã tốn `len(varint(int(m)))` byte. Khoảng cách giữa hai dạng còn
   bao nhiêu?
2. Ở ô `skew_train`, đổi `CANH_THO` thành `112` (ảnh gốc lớn hơn, thu nhỏ mạnh hơn) rồi chạy lại phần 6.
   Những cách tiền xử lý nào giờ làm mất nhiều độ chính xác hơn, và kiểm thử đối chiếu có đổi kết luận không?
3. Ở ô `autoscale`, đổi `PHAT_HIEN` thành `1.0` rồi `15.0`. Với nhóm bản sao chờ sẵn, phần yêu cầu kịp hạn
   phụ thuộc vào thời gian phát hiện ra sao? Một nhóm bản sao chờ sẵn 8 chiếc T4 tốn bao nhiêu mỗi ngày theo
   giá của bảng 22?
4. Ở ô `calib`, gọi `luong_tu(..., bit=4)` (sửa giá trị mặc định của `bit`). Tập hiệu chuẩn nào giờ làm mất
   nhiều nhất?
5. Ở ô `cost_plot`, đổi `RHO` thành `0.8`. Điểm giao nhau giữa CPU và T4 dời về đâu?

## Tóm lại

* Với 1,000 số thực, JSON nặng hơn Protobuf và chậm hơn nhiều khi phân tích; napkin math 1.1 đổi 45 μs mỗi
  yêu cầu ra 45 % một lõi ở 10,000 yêu cầu mỗi giây. Kích thước JSON còn tuỳ cách phía gửi in số.
* Trong bảng 3, tiền xử lý chiếm 44.6 % một yêu cầu ResNet-50; mô hình nhanh gấp 10 lần chỉ cho cả yêu cầu
  nhanh 1.8 lần. Tỉ lệ giữa các pha là của phần cứng, nên phải đo trên đúng máy sẽ phục vụ.
* Bảng 6 dùng hai hằng số không in ra, khoảng 4.1 tỷ phép nhân-cộng và 51 MB trọng số FP16 của ResNet-50;
  tính 2 FLOP cho mỗi phép nhân-cộng thì mọi mật độ trong bảng nhân đôi.
* Pipeline đưa chu kỳ hoạt động của GPU từ 50 % lên gần 100 % khi CPU theo kịp; nếu không, hệ nghẽn ở CPU.
* Lệch giữa huấn luyện và phục vụ không gây lỗi nào; dùng chung hàm biến đổi và một kiểm thử đối chiếu bắt
  được nó trước khi độ chính xác giảm.
* Ánh xạ bộ nhớ dời chi phí nạp sang những yêu cầu đầu; khởi động nguội 35 giây làm hỏng cả một đợt tăng
  vọt, còn engine biên dịch sẵn và nhóm bản sao chờ sẵn rút nó về thời gian phát hiện.
* Bảng 19 và 20 đổi môi trường chạy và độ chính xác ra tăng tốc, và thông lượng gấp ba đổi 30 GPU ra 10.
* Chi phí mỗi lần suy luận tỉ lệ nghịch với mức sử dụng; theo bảng 22, T4 rẻ nhất mỗi ảnh, và cặp 8 V100
  với 24 T4 của chương khớp với ngưỡng mức sử dụng 54 %.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Phục vụ mô hình — mỗi yêu cầu một ngân sách độ trễ](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch13-model-serving/), dựng từ chương
[*Model Serving*](https://mlsysbook.ai/vol1/model_serving/model_serving.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_13_model_serving.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.